In [ ]:
# GPU check
!nvidia-smi


In [ ]:
# Install dependencies
!pip install -q --force-reinstall "transformers>=4.56,<5.0" "kvpress==0.5.3" accelerate datasets nvidia-ml-py bitsandbytes


In [ ]:
# Imports + NVML init
import gc, json, time, threading, statistics
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Optional

import torch
import pynvml
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset

pynvml.nvmlInit()
NVML_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)
print('GPU:', pynvml.nvmlDeviceGetName(NVML_HANDLE))
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())


In [ ]:
# HF login (uncomment if using gated Llama models)


In [ ]:
# Config — models, methods, sweep variables, T4 caps
MODELS = [
    {'name': 'Qwen/Qwen2.5-0.5B-Instruct', 'dtype': torch.float16, 'bnb4': False},
]

METHODS = ['baseline', 'streamingllm', 'snapkv']
TASKS = ['wikitext', 'narrativeqa', 'cnn_dailymail', 'mt_bench']

# §5.3 experimental sweep
CONTEXT_LENGTHS = [2048, 8192, 16384, 32768]
BATCH_SIZES     = [1, 4, 8]
OUTPUT_LENGTHS  = [50, 100, 200]

# T4 reality cap — only ctx=2K supports batching, 16K/32K skipped entirely.
MAX_BS_PER_CTX = {2048: 8, 8192: 1, 16384: 0, 32768: 0}

COMPRESSION_RATIO = 0.5
NUM_WARMUP = 1
NUM_RUNS   = 5
PPL_STRIDE = 512
PPL_SEQ_LEN = 2048
PPL_MAX_CHUNKS = 10
PPL_SPLIT = 'validation'

RESULTS_PATH = Path('results.json')
print('Models :', [m['name'] for m in MODELS])
print('Methods:', METHODS)
print('Tasks  :', TASKS)


In [ ]:
# NVML energy meter + idle baseline measurement
class EnergyMeter:
    def __init__(self, handle, interval=0.1):
        self.handle = handle
        self.interval = interval
        self._samples = []
        self._thread = None
        self._running = False

    def _loop(self):
        while self._running:
            try:
                w = pynvml.nvmlDeviceGetPowerUsage(self.handle) / 1000.0
                self._samples.append((time.time(), w))
            except pynvml.NVMLError:
                pass
            time.sleep(self.interval)

    def __enter__(self):
        self._samples = []
        self._running = True
        self._t0 = time.time()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        self._running = False
        self._thread.join()
        self._t1 = time.time()

    def summary(self, idle_watts=0.0):
        if len(self._samples) < 2:
            return {'energy_j': float('nan'), 'avg_power_w': float('nan'),
                    'peak_power_w': float('nan'), 'n_samples': len(self._samples),
                    'duration_s': self._t1 - self._t0}
        ts = [s[0] for s in self._samples]
        ws = [s[1] - idle_watts for s in self._samples]
        energy = 0.0
        for i in range(1, len(ts)):
            dt = ts[i] - ts[i-1]
            energy += 0.5 * (ws[i] + ws[i-1]) * dt  # trapezoid
        return {
            'energy_j': energy,
            'avg_power_w': sum(ws) / len(ws),
            'peak_power_w': max(ws),
            'n_samples': len(self._samples),
            'duration_s': self._t1 - self._t0,
        }

def measure_idle_watts(seconds=3.0):
    torch.cuda.empty_cache()
    with EnergyMeter(NVML_HANDLE) as m:
        time.sleep(seconds)
    s = m.summary()
    return s['avg_power_w']

IDLE_WATTS = measure_idle_watts()
print(f'Idle GPU power: {IDLE_WATTS:.2f} W')


In [ ]:
# Model loader (SDPA attention) + unload helper
def load_model(cfg):
    kwargs = {'torch_dtype': cfg['dtype'], 'device_map': 'cuda:0'}
    if cfg.get('bnb4'):
        kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
        )
        kwargs.pop('torch_dtype')
    tok = AutoTokenizer.from_pretrained(cfg['name'])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(cfg['name'], **kwargs)
    model.eval()
    return model, tok

def unload(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


In [ ]:
# Compression method factory (baseline / streamingllm / h2o / snapkv)
from contextlib import contextmanager
from kvpress import SnapKVPress, StreamingLLMPress, ObservedAttentionPress

@contextmanager
def noop_press(model):
    yield

def make_press(method, ratio):
    if method == 'baseline':
        return None
    if method == 'h2o':
        return ObservedAttentionPress(compression_ratio=ratio)
    if method == 'streamingllm':
        return StreamingLLMPress(compression_ratio=ratio, n_sink=4)
    if method == 'snapkv':
        return SnapKVPress(compression_ratio=ratio, window_size=64)
    raise ValueError(method)

def with_press(model, press):
    return noop_press(model) if press is None else press(model)


In [ ]:
# Prompt builders, one per §5.4 task
def _wikitext_prompt(tokenizer, target_tokens):
    ds = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train')
    text = '\n\n'.join(t for t in ds['text'][:3000] if t.strip())
    ids = tokenizer(text, return_tensors='pt')['input_ids'][0][:target_tokens - 20]
    return tokenizer.decode(ids) + '\n\nSummary:'

def _narrativeqa_prompt(tokenizer, target_tokens):
    ds = load_dataset('deepmind/narrativeqa', split='validation',
                      streaming=True, trust_remote_code=True)
    ex = next(iter(ds))
    story = ex['document']['text']
    q = ex['question']['text']
    suffix = f'\n\nQuestion: {q}\nAnswer:'
    suffix_len = len(tokenizer(suffix)['input_ids'])
    story_ids = tokenizer(story, return_tensors='pt')['input_ids'][0][:max(1, target_tokens - suffix_len)]
    return tokenizer.decode(story_ids) + suffix

def _cnn_dm_prompt(tokenizer, target_tokens):
    ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split='test', streaming=True)
    pieces, total = [], 0
    for ex in ds:
        ids = tokenizer(ex['article'], return_tensors='pt')['input_ids'][0]
        pieces.append(ids)
        total += len(ids)
        if total >= target_tokens:
            break
    cat = torch.cat(pieces)[:max(1, target_tokens - 10)]
    return tokenizer.decode(cat) + '\n\nSummary:'

def _mt_bench_prompt(tokenizer, target_tokens):
    ds = load_dataset('lmsys/mt_bench_human_judgments', split='human', streaming=True)
    pieces, total = [], 0
    for ex in ds:
        for turn in ex['conversation_a']:
            chunk = f"{turn['role']}: {turn['content']}\n\n"
            ids = tokenizer(chunk, return_tensors='pt')['input_ids'][0]
            pieces.append(ids)
            total += len(ids)
            if total >= target_tokens:
                break
        if total >= target_tokens:
            break
    cat = torch.cat(pieces)[:max(1, target_tokens - 10)]
    return tokenizer.decode(cat) + '\nassistant:'

_PROMPT_BUILDERS = {
    'wikitext': _wikitext_prompt,
    'narrativeqa': _narrativeqa_prompt,
    'cnn_dailymail': _cnn_dm_prompt,
    'mt_bench': _mt_bench_prompt,
}

def build_prompt(task, tokenizer, target_tokens):
    return _PROMPT_BUILDERS[task](tokenizer, target_tokens)

# One prompt per (task, context length); bench cell iterates these.
CONTEXT_LENGTHS = [512, 2048]  # raise to [2048, 8192, 16384] on bigger GPU


In [ ]:
# Per-cell measurement: warmup + measured runs, energy/throughput/memory
@torch.no_grad()
def run_one(model, tokenizer, prompt, method, ratio, max_new_tokens, batch_size, warmup, runs):
    prompts = [prompt] * batch_size
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, truncation=False).to('cuda:0')
    prompt_len = inputs['input_ids'].shape[1]
    press = make_press(method, ratio)

    for _ in range(warmup):
        with with_press(model, press):
            _ = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                do_sample=False, pad_token_id=tokenizer.pad_token_id)
    torch.cuda.synchronize()

    runs_out = []
    for r in range(runs):
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        with EnergyMeter(NVML_HANDLE) as meter:
            t0 = time.time()
            with with_press(model, press):
                out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                      do_sample=False, pad_token_id=tokenizer.pad_token_id)
            torch.cuda.synchronize()
            t1 = time.time()
        es = meter.summary(idle_watts=IDLE_WATTS)
        gen_per_seq = out.shape[1] - prompt_len
        total_gen = gen_per_seq * batch_size
        peak_mem_gb = torch.cuda.max_memory_allocated() / 1024**3
        runs_out.append({
            'run': r,
            'prompt_tokens': prompt_len,
            'batch_size': batch_size,
            'gen_tokens_per_seq': int(gen_per_seq),
            'total_gen_tokens': int(total_gen),
            'latency_s': t1 - t0,
            'throughput_tps': total_gen / (t1 - t0),
            'energy_j': es['energy_j'],
            'j_per_token': es['energy_j'] / total_gen if total_gen > 0 else float('nan'),
            'tokens_per_j': total_gen / es['energy_j'] if es['energy_j'] > 0 else float('nan'),
            'avg_power_w': es['avg_power_w'],
            'peak_power_w': es['peak_power_w'],
            'n_samples': es['n_samples'],
            'peak_mem_gb': peak_mem_gb,
        })
    return runs_out


In [ ]:
# Per-cell measurement: warmup + measured runs, energy/throughput/memory
@torch.no_grad()
def perplexity(model, tokenizer, method, ratio, seq_len=PPL_SEQ_LEN,
               stride=PPL_STRIDE, max_chunks=PPL_MAX_CHUNKS, split=PPL_SPLIT):
    ds = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split=split)
    text = '\n\n'.join(t for t in ds['text'] if t.strip())
    ids = tokenizer(text, return_tensors='pt')['input_ids'].to('cuda:0')
    press = make_press(method, ratio)

    nlls = []
    prev_end = 0
    chunks = 0
    for begin in range(0, ids.size(1), stride):
        end = min(begin + seq_len, ids.size(1))
        trg_len = end - prev_end
        input_ids = ids[:, begin:end]
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100
        with with_press(model, press):
            out = model(input_ids, labels=target_ids)
        nlls.append(out.loss.float() * trg_len)
        prev_end = end
        chunks += 1
        if end == ids.size(1) or chunks >= max_chunks:
            break
    ppl = torch.exp(torch.stack(nlls).sum() / prev_end).item()
    return ppl


In [ ]:
# Load model once + define bench_ctx helper, resume from results.json
all_results = []
if RESULTS_PATH.exists():
    all_results = json.loads(RESULTS_PATH.read_text())
    print(f'Loaded {len(all_results)} existing rows from {RESULTS_PATH}')

ppl_cache = {}

_mcfg = MODELS[0]
print(f'\n=== Loading {_mcfg["name"]} ===')
model, tok = load_model(_mcfg)
print('Model loaded. attn_impl:', getattr(model.config, '_attn_implementation', '?'))

def already_done(model_name, method, task, ctx, bs, out_len):
    return any(r['model'] == model_name and r['method'] == method
               and r.get('task') == task and r['context_len'] == ctx
               and r.get('batch_size') == bs and r.get('output_len') == out_len
               for r in all_results)

def _cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

def bench_ctx(ctx):
    """Run baseline sweep at one context length: tasks x bs x out_len.
    Skips bs values above MAX_BS_PER_CTX[ctx] silently to avoid wasted attempts."""
    mcfg = _mcfg
    model_name = mcfg['name']
    bs_cap = MAX_BS_PER_CTX.get(ctx, max(BATCH_SIZES))
    print(f'\n--- ctx={ctx} (bs_cap={bs_cap}) ---')
    for task in TASKS:
        try:
            prompt = build_prompt(task, tok, target_tokens=ctx)
        except Exception as e:
            print(f'  prompt build failed for {task}@{ctx}: {e!r}')
            continue
        for bs in BATCH_SIZES:
            if bs > bs_cap:
                continue   # silent skip, see MAX_BS_PER_CTX
            for out_len in OUTPUT_LENGTHS:
                for method in METHODS:
                    if already_done(model_name, method, task, ctx, bs, out_len):
                        print(f'  skip {task}/{method} ctx={ctx} bs={bs} out={out_len} (done)')
                        continue
                    print(f'  {task}/{method} ctx={ctx} bs={bs} out={out_len} ...', end=' ', flush=True)
                    _cleanup()
                    try:
                        runs = run_one(model, tok, prompt, method, COMPRESSION_RATIO,
                                       out_len, bs, NUM_WARMUP, NUM_RUNS)
                        pkey = (model_name, method)
                        if pkey not in ppl_cache:
                            ppl_cache[pkey] = perplexity(model, tok, method, COMPRESSION_RATIO)
                        ppl = ppl_cache[pkey]
                    except torch.cuda.OutOfMemoryError:
                        print('OOM (skipped)')
                        _cleanup()
                        continue
                    except Exception as e:
                        print('FAILED:', repr(e))
                        _cleanup()
                        continue
                    row = {
                        'model': model_name,
                        'method': method,
                        'task': task,
                        'context_len': ctx,
                        'batch_size': bs,
                        'output_len': out_len,
                        'compression_ratio': COMPRESSION_RATIO,
                        'idle_watts': IDLE_WATTS,
                        'perplexity': ppl,
                        'runs': runs,
                    }
                    all_results.append(row)
                    RESULTS_PATH.write_text(json.dumps(all_results, indent=2))
                    mean_j   = statistics.mean(r['j_per_token']    for r in runs)
                    mean_tpj = statistics.mean(r['tokens_per_j']   for r in runs)
                    mean_tps = statistics.mean(r['throughput_tps'] for r in runs)
                    mean_mem = statistics.mean(r['peak_mem_gb']    for r in runs)
                    print(f'ppl={ppl:.2f} j/tok={mean_j:.4f} tok/J={mean_tpj:.1f} tps={mean_tps:.1f} mem={mean_mem:.2f}GB')
    print(f'  ctx={ctx} done. Total rows: {len(all_results)}')


In [ ]:
# Sweep at ctx=2K
bench_ctx(2048)


In [ ]:
# Sweep at ctx=8K (T4 cap: bs=1 only)
bench_ctx(8192)


In [ ]:
# Sweep at ctx=16K (T4 cap: skipped, lab GPU only)
bench_ctx(16384)


In [ ]:
# Sweep at ctx=32K (T4 cap: skipped, lab GPU only)
bench_ctx(32768)


In [ ]:
# Unload model from GPU
unload(model)
print('model unloaded')


In [ ]:
# Summary dataframe with all §5 dependent variables + comparison columns
import pandas as pd
rows = []
for r in all_results:
    j   = [x['j_per_token']    for x in r['runs']]
    tpj = [x['tokens_per_j']   for x in r['runs']]
    t   = [x['throughput_tps'] for x in r['runs']]
    m   = [x['peak_mem_gb']    for x in r['runs']]
    rows.append({
        'model':         r['model'].split('/')[-1],
        'task':          r.get('task', 'wikitext'),
        'method':        r['method'],
        'ctx':           r['context_len'],
        'bs':            r.get('batch_size', 1),
        'out':           r.get('output_len', 200),
        'ppl':           round(r['perplexity'], 2),
        'j_per_tok':     round(np.mean(j), 4),
        'j_per_tok_std': round(np.std(j), 4),
        'tok_per_j':     round(np.mean(tpj), 2),
        'tps':           round(np.mean(t), 1),
        'peak_mem_gb':   round(np.mean(m), 2),
    })
df = pd.DataFrame(rows).sort_values(['model', 'task', 'ctx', 'bs', 'out', 'method'])

# Perplexity is task-independent (always WikiText-2 val). Key on (model, ctx).
_base_ppl = (df[df['method'] == 'baseline']
             .drop_duplicates(['model', 'ctx'])
             .set_index(['model', 'ctx'])['ppl'])
df['ppl_delta_pct'] = df.apply(
    lambda r: round(100 * (r['ppl'] - _base_ppl.get((r['model'], r['ctx']), np.nan))
                    / _base_ppl.get((r['model'], r['ctx']), np.nan), 2)
              if (r['model'], r['ctx']) in _base_ppl.index else np.nan,
    axis=1)

# Energy depends on the prompt and shape. Key on (model, task, ctx, bs, out).
_base_j = (df[df['method'] == 'baseline']
           .set_index(['model', 'task', 'ctx', 'bs', 'out'])['j_per_tok'])
df['j_savings_pct'] = df.apply(
    lambda r: round(100 * (_base_j.get((r['model'], r['task'], r['ctx'], r['bs'], r['out']), np.nan) - r['j_per_tok'])
                    / _base_j.get((r['model'], r['task'], r['ctx'], r['bs'], r['out']), np.nan), 2)
              if (r['model'], r['task'], r['ctx'], r['bs'], r['out']) in _base_j.index else np.nan,
    axis=1)

df


In [ ]:
# Comparison tables — pivots by method, ctx, task
from IPython.display import display

n_methods = df['method'].nunique()

print('=' * 70)
print(f'Rows: {len(df)} | Methods: {sorted(df["method"].unique())} | Tasks: {sorted(df["task"].unique())}')
print('=' * 70)

if n_methods == 1:
    print('\n--- Energy per token (J/tok) by (task, ctx) at bs=1, out=200 ---')
    sub = df[(df['bs'] == 1) & (df['out'] == 200)]
    pivot = sub.pivot_table(index='task', columns='ctx', values='j_per_tok')
    display(pivot.round(3))

    print('\n--- Energy per token (J/tok) by (bs, out) at ctx=2048 wikitext ---')
    sub = df[(df['task'] == 'wikitext') & (df['ctx'] == 2048)]
    pivot = sub.pivot_table(index='bs', columns='out', values='j_per_tok')
    display(pivot.round(3))

    print('\n--- Throughput (tok/s) by (bs, out) at ctx=2048 wikitext ---')
    pivot = sub.pivot_table(index='bs', columns='out', values='tps')
    display(pivot.round(1))

    print('\n--- Peak memory (GB) by (task, ctx) at bs=1, out=200 ---')
    sub = df[(df['bs'] == 1) & (df['out'] == 200)]
    pivot = sub.pivot_table(index='task', columns='ctx', values='peak_mem_gb')
    display(pivot.round(2))

else:
    print('\n--- Energy savings vs baseline (%) by method, ctx, task at bs=1 out=200 ---')
    sub = df[(df['bs'] == 1) & (df['out'] == 200) & (df['method'] != 'baseline')]
    pivot = sub.pivot_table(index=['method', 'task'], columns='ctx', values='j_savings_pct')
    display(pivot.round(1))

    print('\n--- Perplexity delta (%) by method, ctx ---')
    sub = df[df['method'] != 'baseline']
    pivot = sub.pivot_table(index='method', columns='ctx', values='ppl_delta_pct')
    display(pivot.round(2))

    print('\n--- Per-method summary at ctx=2048, bs=1, out=200 ---')
    sub = df[(df['ctx'] == 2048) & (df['bs'] == 1) & (df['out'] == 200)]
    cols = ['method', 'task', 'j_per_tok', 'tps', 'peak_mem_gb', 'ppl', 'j_savings_pct', 'ppl_delta_pct']
    display(sub[cols].sort_values(['task', 'method']).reset_index(drop=True))

# Always save a CSV next to results.json for offline analysis / Excel / external plotting.
df.to_csv('summary.csv', index=False)
print(f'\nFull summary written to summary.csv ({len(df)} rows). Download from the file panel.')
